# Phase 4 YOLO Unified Evaluation

Export the frozen YOLO baseline on the Phase 3 validation split and evaluate it with the framework-independent COCO/FROC protocol.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import pandas as pd

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True, check=False,
).stdout.strip()
print('GPU:', gpu_name or '<none>')
print('INPUT ROOTS:', sorted(str(path) for path in Path('/kaggle/input').iterdir()))

## 1. Install the frozen runtime

In [ ]:
if 'P100' in gpu_name:
    !pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

!pip install --no-cache-dir ultralytics==8.4.104 ensemble-boxes pycocotools==2.0.10 tqdm

## 2. Load the exact project revision

In [ ]:
%cd /kaggle/working
repo = Path('cxr-detectbench')
if repo.exists():
    shutil.rmtree(repo)
!git clone https://github.com/kinjazA/cxr-detectbench.git
%cd /kaggle/working/cxr-detectbench
!git rev-parse --short HEAD

## 3. Discover and validate mounted inputs

In [ ]:
required_train_columns = {
    'image_id', 'class_name', 'class_id', 'rad_id',
    'x_min', 'y_min', 'x_max', 'y_max',
}
train_csv_candidates = []
for candidate in Path('/kaggle/input').rglob('train.csv'):
    try:
        columns = set(pd.read_csv(candidate, nrows=1).columns)
    except Exception:
        continue
    if required_train_columns.issubset(columns):
        train_csv_candidates.append(candidate)
if len(train_csv_candidates) != 1:
    raise RuntimeError(f'Expected one schema-matching train.csv, got: {train_csv_candidates}')
train_csv = train_csv_candidates[0]

train_meta = Path('/kaggle/input/datasets/corochann/vinbigdata-chest-xray-original-png/train_meta.csv')
train_png_dir = Path('/kaggle/input/datasets/corochann/vinbigdata-chest-xray-original-png/train')
if not train_meta.is_file() or not train_png_dir.is_dir():
    raise FileNotFoundError(f'Missing confirmed PNG inputs: {train_meta}, {train_png_dir}')
if len(list(train_png_dir.glob('*.png'))) != 15000:
    raise RuntimeError('Confirmed PNG train directory does not contain 15,000 files')

best_candidates = [
    path for path in Path('/kaggle/input').rglob('best.pt')
    if path.parent.name == 'weights'
]
if len(best_candidates) != 1:
    raise RuntimeError(f'Expected one mounted baseline best.pt, got: {best_candidates}')
best_weight = best_candidates[0]
print('train_csv:', train_csv)
print('train_meta:', train_meta)
print('train_png_dir:', train_png_dir)
print('best_weight:', best_weight)

## 4. Rebuild the frozen validation dataset

In [ ]:
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, 'data/raw/train.csv')
shutil.copy2(train_meta, 'data/raw/images.csv')
images_link = Path('data/processed/images_png')
if images_link.is_symlink() or images_link.is_file():
    images_link.unlink()
elif images_link.exists():
    shutil.rmtree(images_link)
images_link.symlink_to(train_png_dir, target_is_directory=True)

!python scripts/label_fusion.py --train_csv data/raw/train.csv --images_csv data/raw/images.csv --output_dir data/processed/labels_coco --iou_thr 0.5
!python scripts/prepare_yolo_dataset.py --coco_json data/processed/labels_coco/wbf/annotations.json --images_dir data/processed/images_png --splits_dir data/splits --output_dir data/processed/yolo_wbf_phase3 --overwrite

## 5. Export predictions and run the shared evaluator

In [ ]:
summary_dir = Path('/kaggle/working/yolo_unified_eval_summary')
summary_dir.mkdir(parents=True, exist_ok=True)
predictions_path = Path('/kaggle/working/yolo_val_predictions.json')

!python scripts/export_ultralytics_predictions.py \
    --model {best_weight} \
    --images-dir data/processed/yolo_wbf_phase3/images/val \
    --split-csv data/splits/val.csv \
    --output {predictions_path} \
    --summary {summary_dir / 'inference_summary.json'} \
    --imgsz 640 --batch 16 --confidence 0.001 --nms-iou 0.7 --max-detections 300

!python scripts/evaluate_detection.py \
    --ground-truth data/processed/labels_coco/wbf/annotations.json \
    --predictions {predictions_path} \
    --split-csv data/splits/val.csv \
    --output-dir {summary_dir} \
    --max-detections 100 --froc-iou 0.5

## 6. Print compact results and clean regenerated data

In [ ]:
print((summary_dir / 'metrics.json').read_text())
print((summary_dir / 'per_class_metrics.csv').read_text())
print((summary_dir / 'inference_summary.json').read_text())
print('SUMMARY FILES:')
for path in sorted(summary_dir.iterdir()):
    print(path.name, path.stat().st_size, 'bytes')

if predictions_path.exists():
    predictions_path.unlink()
for path in [Path('/kaggle/working/cxr-detectbench')]:
    if path.exists():
        shutil.rmtree(path)
print('Cleaned predictions and regenerated project data')